<a href="https://colab.research.google.com/github/kumarshree-naik/data-engineering-portfolio/blob/main/project2_pyspark_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TaxiBigData") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark is ready!")

In [ ]:
# Download 3 months of NYC taxi data
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet
print("Downloaded. Files:")
!ls -lh *.parquet

In [ ]:
# Spark reads multiple files at once with a wildcard
df = spark.read.parquet("yellow_tripdata_2024-*.parquet")

# Count rows — this is an ACTION, it triggers actual computation
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

In [ ]:
df.printSchema()
df.show(5)

In [ ]:
from pyspark.sql import functions as F

# --- CLEAN ---
clean_df = df \
    .filter(F.col("fare_amount") > 0) \
    .filter(F.col("trip_distance") > 0) \
    .filter(F.col("total_amount") > 0) \
    .filter(F.col("passenger_count") > 0) \
    .filter(F.col("tpep_pickup_datetime") >= "2024-01-01") \
    .filter(F.col("tpep_pickup_datetime") < "2024-04-01")

# --- ADD DERIVED COLUMNS (feature engineering) ---
clean_df = clean_df \
    .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") -
                 F.unix_timestamp("tpep_pickup_datetime")) / 60) \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day", F.date_format("tpep_pickup_datetime", "EEEE"))

# Keep only trips with positive duration
clean_df = clean_df.filter(F.col("trip_duration_min") > 0)

print("Rows after cleaning:", clean_df.count())

In [ ]:
from pyspark.sql import functions as F

# Aggregation 1: Trips and revenue by hour of day
print("=== Trips & avg fare by pickup hour ===")
clean_df.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min")
    ) \
    .orderBy("pickup_hour") \
    .show(24)

In [ ]:
clean_df.groupBy("pickup_day") \
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    ) \
    .orderBy(F.desc("trips")) \
    .show()

In [ ]:
clean_df.withColumn("distance_bucket",
        F.when(F.col("trip_distance") < 2, "0-2 mi")
         .when(F.col("trip_distance") < 5, "2-5 mi")
         .when(F.col("trip_distance") < 10, "5-10 mi")
         .otherwise("10+ mi")) \
    .groupBy("distance_bucket") \
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    ) \
    .show()

In [ ]:
# Save the hourly aggregation as a table we'll push to BigQuery
hourly_agg = clean_df.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min")
    ) \
    .orderBy("pickup_hour")

# Convert Spark result to Pandas (it's small — just 24 rows — so this is safe)
hourly_pd = hourly_agg.toPandas()
print(hourly_pd)